# Train a Weapon-Detection YOLOv8 Model (Google Colab)

Fine-tune YOLOv8 on a gun/knife dataset using Colab's free GPU, then download `best.pt` for the Real-Time Weapon Detection app.

**Steps:** enable a GPU runtime (Runtime -> Change runtime type -> T4 GPU), then run the cells top to bottom.

You need a free Roboflow API key: https://app.roboflow.com -> Settings -> API.

## 1. Confirm a GPU is available

In [ ]:
!nvidia-smi

## 2. Install dependencies

In [ ]:
!pip install -q ultralytics roboflow

## 3. Download a labelled weapon dataset from Roboflow Universe

This defaults to the public **`joseph-nelson/pistols`** dataset (~2,970 labelled pistol images, class name `pistol`), which is verified to download in YOLOv8 format.

To use a different dataset, browse https://universe.roboflow.com, open one, click **Download Dataset -> YOLOv8 -> show download code**, and replace `WORKSPACE`, `PROJECT`, `VERSION` below. Whatever you pick, set `model.weapon_classes` in your app config to match the dataset's class names (printed by `data.yaml`).

In [ ]:
from roboflow import Roboflow

ROBOFLOW_API_KEY = "PASTE_YOUR_API_KEY"  # https://app.roboflow.com -> Settings -> API
WORKSPACE = "joseph-nelson"  # verified public pistol dataset
PROJECT = "pistols"
VERSION = 1  # dataset version number

import random, shutil, yaml
from pathlib import Path

rf = Roboflow(api_key=ROBOFLOW_API_KEY)
dataset = rf.workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov8")
location = Path(dataset.location)
print("Dataset at:", location)

# Some Roboflow versions export a flat folder with no train/val split; build one.
def _imgs(d):
    d = Path(d)
    return [p for p in d.iterdir() if p.suffix.lower() in {'.jpg','.jpeg','.png','.bmp','.webp'}] if d.is_dir() else []

if not (_imgs(location/'train'/'images') and _imgs(location/'valid'/'images')):
    pool = next((d for d in (location/'export'/'images', location/'images') if _imgs(d)), None)
    if pool is not None:
        labels = pool.parent/'labels'
        imgs = sorted(_imgs(pool)); random.seed(42); random.shuffle(imgs)
        nval = max(1, int(len(imgs)*0.15)); val = set(imgs[:nval])
        for s in ('train','valid'):
            (location/s/'images').mkdir(parents=True, exist_ok=True)
            (location/s/'labels').mkdir(parents=True, exist_ok=True)
        for img in imgs:
            s = 'valid' if img in val else 'train'
            shutil.copy(img, location/s/'images'/img.name)
            lb = labels/(img.stem+'.txt')
            if lb.exists(): shutil.copy(lb, location/s/'labels'/lb.name)
        y = yaml.safe_load((location/'data.yaml').read_text()) or {}
        y['train'], y['val'] = 'train/images', 'valid/images'; y.pop('test', None)
        (location/'data.yaml').write_text(yaml.safe_dump(y, sort_keys=False))
        print(f'Split flat dataset: {len(imgs)-nval} train / {nval} val')

## 4. Train

50 epochs on a T4 GPU typically takes tens of minutes depending on dataset size. Start with `yolov8n.pt` (fast); use `yolov8s.pt`/`yolov8m.pt` for higher accuracy.

In [ ]:
from ultralytics import YOLO

model = YOLO("yolov8n.pt")
results = model.train(
    data=f"{dataset.location}/data.yaml",
    epochs=50,
    imgsz=640,
    batch=16,
    name="weapon_yolov8",
)

## 5. Validate (precision / recall / mAP)

In [ ]:
metrics = model.val()
print(metrics)

## 6. Download best.pt

Save `best.pt` and copy it into your project (e.g. `models/best.pt`), then run:

```
weapon-detector --weights models/best.pt
```

or set `model.weights: models/best.pt` in your config.

In [ ]:
from google.colab import files

best = f"{results.save_dir}/weights/best.pt"
print("Best weights:", best)
files.download(best)